In [ ]:
# OBELiX AUDIT-FILTERED PUBLICATION ANALYSIS — SINGLE GOOGLE COLAB CELL
#
# This driver performs the complete reviewer-response analysis using only
# audit-eligible spectra. It preserves the existing input paths, creates
# read-only filtered roots under /content, runs the repository's mature
# association/kernel/reviewer engine with all reviewer analyses enabled,
# suppresses its legacy figures, and then exports publication figures for
# which every PDF/PNG has a matching CSV containing the exact plotted data.

NOTEBOOK_VERSION = "2026-07-19-v5-complete-supplement-reference"
print(f"OBELiX analysis notebook version: {NOTEBOOK_VERSION}")

# =============================== CONFIGURATION ===============================
from pathlib import Path

TRAIN_PDOS_ROOT = Path("/content/drive/MyDrive/MLIP-main/DOSFILES/Train_cif")
TEST_PDOS_ROOT  = Path("/content/drive/MyDrive/MLIP-main/DOSFILES/Test_cif")
AUDIT_CSV = Path(
    "/content/drive/MyDrive/OBELiX_composition_audit/"
    "tables/production_structure_composition_audit.csv"
)
MASTER_SCRIPT = Path(
    "/content/drive/MyDrive/MLIP-main/"
    "obelix_reviewer_response_onecell_code_2.py"
)
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/OBELiX_convergence_outputs/"
    "audit_filtered_publication_analysis"
)

# Audit cohorts.
PRIMARY_COMPOSITION_STATUSES = {
    "PASS",
    "PASS_INTEGERIZED_COMPOSITION",
    "WARN_INTEGERIZATION_APPROXIMATION",
}
STRICT_COMPOSITION_STATUSES = {
    "PASS",
    "PASS_INTEGERIZED_COMPOSITION",
}
EXACT_COMPOSITION_STATUSES = {"PASS"}
INCLUDE_SHORT_DISTANCE_WARNINGS = False

# Statistical settings for the archival run.
ASSOCIATION_BOOTSTRAP = 2000
ASSOCIATION_PERMUTATIONS = 5000
ASSOCIATION_MI_PERMUTATIONS = 500
KERNEL_BOOTSTRAP = 5000
HSIC_PERMUTATIONS = 5000
REVIEWER_BOOTSTRAP = 5000
RANDOM_SEED = 20260718
FIGURE_DPI = 600

# Optional expensive calculations. They execute only when a real manifest is
# found; placeholder/template manifests are ignored.
RUN_REFERENCE_VALIDATION = True
RUN_PHONON_CONVERGENCE_IF_MANIFEST_EXISTS = True

# =========================== INSTALL AND MOUNT DRIVE =========================
import os, sys, re, json, math, shutil, hashlib, zipfile, warnings, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas>=2.1", "numpy>=1.26", "scipy>=1.11", "matplotlib>=3.8",
    "scikit-learn>=1.4", "statsmodels>=0.14", "dcor>=0.6",
    "pymatgen>=2025.6.14", "pyyaml>=6.0", "xgboost>=2.0",
])

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.axes import Axes
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_SEED)

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

for required_path, label in [
    (TRAIN_PDOS_ROOT, "training spectrum root"),
    (TEST_PDOS_ROOT, "test spectrum root"),
    (AUDIT_CSV, "revised audit CSV"),
]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing {label}: {required_path}")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVER_TABLES = OUTPUT_ROOT / "driver_tables"
MAIN_FIGURES = OUTPUT_ROOT / "publication_figures" / "main"
SUPP_FIGURES = OUTPUT_ROOT / "publication_figures" / "supplement"
REFERENCE_FIGURES = OUTPUT_ROOT / "publication_figures" / "reference_validation"
PLOT_DATA = OUTPUT_ROOT / "plot_data"
REFERENCE_CASE_DATA = PLOT_DATA / "reference_cases"
SUPPLEMENT_TABLES = OUTPUT_ROOT / "supplement_tables"
REFERENCE_TABLES = OUTPUT_ROOT / "reference_validation_tables"
LOGS = OUTPUT_ROOT / "logs"
for p in [DRIVER_TABLES, MAIN_FIGURES, SUPP_FIGURES, REFERENCE_FIGURES,
          PLOT_DATA, REFERENCE_CASE_DATA, SUPPLEMENT_TABLES,
          REFERENCE_TABLES, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

# Publication style: no titles, large readable labels, vector-compatible fonts.
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 1,
    "xtick.labelsize": 11.5,
    "ytick.labelsize": 11.5,
    "legend.fontsize": 10.5,
    "axes.linewidth": 1.0,
    "lines.linewidth": 1.8,
    "lines.markersize": 6.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": FIGURE_DPI,
    "savefig.bbox": "tight",
})

# ========================== LOAD OFFICIAL METADATA ===========================
TRAIN_METADATA_URL = "https://raw.githubusercontent.com/NRC-Mila/OBELiX/main/data/downloads/train.csv"
TEST_METADATA_URL  = "https://raw.githubusercontent.com/NRC-Mila/OBELiX/main/data/downloads/test.csv"

def normalize_id(x):
    return str(x).strip().lower()

def parse_conductivity(value):
    if pd.isna(value):
        return np.nan, False, ""
    s = str(value).strip().replace("−", "-").replace("×", "x")
    comparator = "<" if "<" in s else (">" if ">" in s else "")
    s = re.sub(r"([0-9.]+)\s*[xX]\s*10\s*\^?\s*([-+]?\d+)", r"\1e\2", s)
    m = re.search(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?", s)
    if not m:
        return np.nan, False, comparator
    try:
        value = float(m.group(0))
    except Exception:
        return np.nan, False, comparator
    return value, comparator == "<", comparator

def load_metadata(url, split):
    df = pd.read_csv(url, dtype={"ID": str})
    df["ID"] = df["ID"].map(normalize_id)
    df["split"] = split
    parsed = df["Ionic conductivity (S cm-1)"].map(parse_conductivity)
    df["ic_reported"] = [x[0] for x in parsed]
    df["ic_is_upper_limit"] = [x[1] for x in parsed]
    df["ic_comparator"] = [x[2] for x in parsed]
    df.loc[df["ic_reported"] <= 0, "ic_reported"] = np.nan
    df["ic_limit"] = df["ic_reported"]
    df["ic_half_limit"] = np.where(
        df["ic_is_upper_limit"], 0.5 * df["ic_reported"], df["ic_reported"]
    )
    df["ic_exclude"] = np.where(df["ic_is_upper_limit"], np.nan, df["ic_reported"])
    for policy in ["limit", "half_limit", "exclude"]:
        df[f"log10_ic_{policy}"] = np.log10(pd.to_numeric(df[f"ic_{policy}"], errors="coerce"))
    if "Family" not in df.columns:
        df["Family"] = "Unknown"
    df["Family"] = df["Family"].fillna("Unknown").astype(str).str.strip().replace("", "Unknown")
    return df

train_metadata = load_metadata(TRAIN_METADATA_URL, "train")
test_metadata = load_metadata(TEST_METADATA_URL, "test")
metadata = pd.concat([train_metadata, test_metadata], ignore_index=True)
metadata = metadata.drop_duplicates(["split", "ID"], keep="first")
known_ids = set(metadata["ID"])
metadata.to_csv(DRIVER_TABLES / "official_metadata_with_parsed_conductivity.csv", index=False)

# ======================== AUDIT-ELIGIBLE COHORTS =============================
audit = pd.read_csv(AUDIT_CSV, dtype={"ID": str})
audit["ID"] = audit["ID"].map(normalize_id)
audit["split"] = audit["split"].astype(str).str.strip().str.lower()

# The revised v3 audit provides composition_status. Fall back to audit_status
# only when composition_status is unavailable.
if "composition_status" not in audit.columns:
    audit["composition_status"] = audit.get("audit_status", "")
audit["composition_status"] = audit["composition_status"].fillna("").astype(str)
if "geometry_status" not in audit.columns:
    audit["geometry_status"] = np.where(
        audit.get("audit_status", "").astype(str).eq("FAIL_SEVERE_OVERLAP"),
        "FAIL_SEVERE_OVERLAP",
        np.where(
            audit.get("audit_status", "").astype(str).eq("WARN_SHORT_DISTANCE"),
            "WARN_SHORT_DISTANCE",
            "PASS",
        ),
    )

def geometry_eligible(series):
    status = series.fillna("PASS").astype(str)
    allowed = ~status.eq("FAIL_SEVERE_OVERLAP")
    if not INCLUDE_SHORT_DISTANCE_WARNINGS:
        allowed &= ~status.eq("WARN_SHORT_DISTANCE")
    return allowed

audit["eligible_primary_audit"] = (
    audit["composition_status"].isin(PRIMARY_COMPOSITION_STATUSES)
    & geometry_eligible(audit["geometry_status"])
)
audit["eligible_strict_audit"] = (
    audit["composition_status"].isin(STRICT_COMPOSITION_STATUSES)
    & geometry_eligible(audit["geometry_status"])
)
audit["eligible_exact_audit"] = (
    audit["composition_status"].isin(EXACT_COMPOSITION_STATUSES)
    & geometry_eligible(audit["geometry_status"])
)
audit.to_csv(DRIVER_TABLES / "audit_with_analysis_eligibility.csv", index=False)

# ===================== ROBUST SPECTRUM-PAIR SELECTION ========================
def exact_pair_candidates(root, split):
    rows = []
    root = Path(root)
    li_files = list(root.rglob("Lithium_PDOS_*.csv"))
    for li_path in li_files:
        m = re.match(r"Lithium_PDOS_([A-Za-z0-9]{3})\.csv$", li_path.name, flags=re.I)
        if not m:
            continue
        oid = normalize_id(m.group(1))
        if oid not in known_ids:
            continue
        total_path = li_path.parent / f"Total_DOS_{oid}.csv"
        if not total_path.exists():
            # Case-insensitive fallback in the same directory only.
            hits = [p for p in li_path.parent.glob("Total_DOS_*.csv") if normalize_id(p.stem.split("_")[-1]) == oid]
            total_path = hits[0] if hits else None
        if total_path is None or not Path(total_path).exists():
            continue
        rel = li_path.parent.relative_to(root)
        folder_exact = normalize_id(li_path.parent.name) == oid
        score = 1000 * int(folder_exact) - 10 * len(rel.parts)
        if "backup" in str(li_path).lower() or "/old" in str(li_path).lower():
            score -= 500
        rows.append({
            "split": split,
            "ID": oid,
            "source_folder": str(li_path.parent),
            "li_file": str(li_path),
            "total_file": str(total_path),
            "selection_score": score,
            "folder_name_exact_ID": folder_exact,
        })
    return pd.DataFrame(rows)

candidate_pairs = pd.concat([
    exact_pair_candidates(TRAIN_PDOS_ROOT, "train"),
    exact_pair_candidates(TEST_PDOS_ROOT, "test"),
], ignore_index=True)
if candidate_pairs.empty:
    raise RuntimeError("No exact Lithium_PDOS_<ID>.csv / Total_DOS_<ID>.csv pairs were found.")
candidate_pairs = candidate_pairs.sort_values(
    ["split", "ID", "selection_score", "source_folder"],
    ascending=[True, True, False, True],
)
candidate_pairs["candidate_rank"] = candidate_pairs.groupby(["split", "ID"]).cumcount() + 1
candidate_pairs["selected"] = candidate_pairs["candidate_rank"].eq(1)
candidate_pairs.to_csv(DRIVER_TABLES / "spectrum_pair_candidates.csv", index=False)
selected_pairs = candidate_pairs[candidate_pairs["selected"]].copy()

cohort_manifest = metadata.merge(
    audit,
    on=["split", "ID"],
    how="left",
    suffixes=("", "_audit"),
).merge(
    selected_pairs[["split", "ID", "source_folder", "li_file", "total_file", "selection_score"]],
    on=["split", "ID"],
    how="left",
)
cohort_manifest["has_paired_spectrum"] = cohort_manifest["li_file"].notna() & cohort_manifest["total_file"].notna()
cohort_manifest["has_target_limit_policy"] = np.isfinite(cohort_manifest["log10_ic_limit"])
for cohort in ["primary", "strict", "exact"]:
    cohort_manifest[f"included_{cohort}"] = (
        cohort_manifest[f"eligible_{cohort}_audit"].fillna(False)
        & cohort_manifest["has_paired_spectrum"]
        & cohort_manifest["has_target_limit_policy"]
    )

reasons = []
for _, r in cohort_manifest.iterrows():
    rr = []
    if not bool(r.get("eligible_primary_audit", False)):
        rr.append(f"audit:{r.get('composition_status', '')}/{r.get('geometry_status', '')}")
    if not bool(r.get("has_paired_spectrum", False)):
        rr.append("missing_paired_spectrum")
    if not bool(r.get("has_target_limit_policy", False)):
        rr.append("missing_conductivity_target")
    reasons.append(";".join(rr))
cohort_manifest["primary_exclusion_reason"] = reasons
cohort_manifest.to_csv(DRIVER_TABLES / "cohort_master_manifest.csv", index=False)
for cohort in ["primary", "strict", "exact"]:
    cohort_manifest[cohort_manifest[f"included_{cohort}"]].to_csv(
        DRIVER_TABLES / f"cohort_included_{cohort}.csv", index=False
    )
cohort_manifest[~cohort_manifest["included_primary"]].to_csv(
    DRIVER_TABLES / "cohort_exclusions.csv", index=False
)

# ================= CREATE LOCAL AUDIT-FILTERED ROOTS =========================
# IMPORTANT: Do not stage material directories as directory symlinks. pathlib's
# rglob(), which is used by the embedded reviewer workflow, does not recurse
# through directory symlinks. Instead, create real local material directories
# under /content and hard-link individual files when the source and destination
# filesystems permit it; otherwise copy the files. Google Drive -> /content is
# normally cross-filesystem, so the safe fallback is shutil.copy2(). The Drive
# folders remain read-only and are never modified.
FILTERED_BASE = Path("/content/obelix_audit_filtered_spectra")
if FILTERED_BASE.exists():
    shutil.rmtree(FILTERED_BASE)
FILTERED_TRAIN = FILTERED_BASE / "Train_cif"
FILTERED_TEST = FILTERED_BASE / "Test_cif"
FILTERED_TRAIN.mkdir(parents=True, exist_ok=True)
FILTERED_TEST.mkdir(parents=True, exist_ok=True)


def materialize_file(source, destination):
    """Copy a real file into /content; never create a directory or file symlink."""
    source = Path(source)
    destination = Path(destination)
    if not source.exists():
        raise FileNotFoundError(f"Required source path does not exist: {source}")
    if not source.is_file():
        raise FileNotFoundError(f"Required source path is not a regular file: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() or destination.is_symlink():
        destination.unlink()
    shutil.copy2(str(source), str(destination))
    if not destination.is_file() or destination.stat().st_size == 0:
        raise IOError(f"Staged file is missing or empty after copy: {destination}")
    return "copy2"


def auxiliary_files_for_staging(source_folder):
    """Yield small provenance/QC auxiliaries without copying heavy force files."""
    source_folder = Path(source_folder)
    seen = set()
    for path in source_folder.rglob("*"):
        if not path.is_file():
            continue
        name = path.name.lower()
        suffix = path.suffix.lower()
        keep = (
            suffix == ".cif"
            or name == "phonon_eigen_data.npz"
            or name in {"poscar", "contcar"}
            or name.startswith("poscar-")
            or name.startswith("contcar-")
            or name in {
                "phonopy_params.yaml", "phonopy_params.yaml.xz",
                "phonopy.yaml", "phonopy.yaml.xz",
            }
        )
        if not keep:
            continue
        # Avoid accidentally staging very large archives. The spectra needed by
        # the analyses are copied separately and never encounter this limit.
        try:
            if path.stat().st_size > 250 * 1024 * 1024:
                continue
        except Exception:
            continue
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            yield path


primary_rows = cohort_manifest[cohort_manifest["included_primary"]].copy()
staging_rows = []
staging_error_rows = []
for _, row in primary_rows.sort_values(["split", "ID"]).iterrows():
    split = str(row["split"])
    oid = normalize_id(row["ID"])
    destination_root = FILTERED_TRAIN if split == "train" else FILTERED_TEST
    destination = destination_root / oid
    source_folder = Path(row["source_folder"])
    destination.mkdir(parents=True, exist_ok=True)

    try:
        # Always use canonical filenames expected by the repository workflow.
        li_destination = destination / f"Lithium_PDOS_{oid}.csv"
        total_destination = destination / f"Total_DOS_{oid}.csv"
        li_method = materialize_file(Path(row["li_file"]), li_destination)
        total_method = materialize_file(Path(row["total_file"]), total_destination)

        auxiliary_count = 0
        auxiliary_methods = []
        for src in auxiliary_files_for_staging(source_folder):
            try:
                rel = src.relative_to(source_folder)
            except Exception:
                rel = Path(src.name)
            target = destination / "auxiliary" / rel
            try:
                auxiliary_methods.append(materialize_file(src, target))
                auxiliary_count += 1
            except Exception as exc:
                staging_error_rows.append({
                    "split": split, "ID": oid, "stage": "auxiliary_staging",
                    "source_file": str(src), "destination_file": str(target),
                    "error": str(exc),
                })

        staging_rows.append({
            "split": split,
            "ID": oid,
            "source_folder": str(source_folder),
            "filtered_folder": str(destination),
            "source_li_file": str(row["li_file"]),
            "source_total_file": str(row["total_file"]),
            "filtered_li_file": str(li_destination),
            "filtered_total_file": str(total_destination),
            "li_staging_method": li_method,
            "total_staging_method": total_method,
            "auxiliary_file_count": auxiliary_count,
            "auxiliary_staging_methods": ";".join(sorted(set(auxiliary_methods))),
            "status": "ok",
        })
    except Exception as exc:
        staging_rows.append({
            "split": split,
            "ID": oid,
            "source_folder": str(source_folder),
            "filtered_folder": str(destination),
            "status": "failed",
            "error": str(exc),
        })
        staging_error_rows.append({
            "split": split, "ID": oid, "stage": "required_spectrum_staging",
            "source_file": str(source_folder), "destination_file": str(destination),
            "error": str(exc),
        })

staging_df = pd.DataFrame(staging_rows)
staging_df.to_csv(DRIVER_TABLES / "filtered_root_staging_manifest.csv", index=False)
pd.DataFrame(staging_error_rows).to_csv(
    DRIVER_TABLES / "filtered_root_staging_errors.csv", index=False
)


def validate_materialized_root(root, split, expected_ids):
    root = Path(root)
    expected_ids = {normalize_id(x) for x in expected_ids}
    li_files = sorted(root.rglob("Lithium_PDOS_*.csv"))
    total_files = sorted(root.rglob("Total_DOS_*.csv"))

    def ids_from(paths, prefix):
        out = set()
        for p in paths:
            m = re.fullmatch(rf"{re.escape(prefix)}_([A-Za-z0-9]{{3}})\.csv", p.name, flags=re.I)
            if m:
                out.add(normalize_id(m.group(1)))
        return out

    li_ids = ids_from(li_files, "Lithium_PDOS")
    total_ids = ids_from(total_files, "Total_DOS")
    paired_ids = li_ids & total_ids
    missing_li = sorted(expected_ids - li_ids)
    missing_total = sorted(expected_ids - total_ids)
    unexpected = sorted(paired_ids - expected_ids)

    validation = pd.DataFrame([{
        "split": split,
        "root": str(root),
        "expected_materials": len(expected_ids),
        "Li_PDOS_files_found": len(li_files),
        "Total_DOS_files_found": len(total_files),
        "paired_materials_found": len(paired_ids),
        "missing_Li_IDs": ";".join(missing_li),
        "missing_total_IDs": ";".join(missing_total),
        "unexpected_paired_IDs": ";".join(unexpected),
    }])
    if missing_li or missing_total or len(paired_ids) != len(expected_ids):
        raise RuntimeError(
            f"Filtered {split} staging failed: expected {len(expected_ids)} paired materials, "
            f"found {len(paired_ids)}. Missing Li IDs={missing_li[:20]}; "
            f"missing total IDs={missing_total[:20]}. See filtered_root_staging_errors.csv."
        )
    return validation


expected_train_ids = primary_rows.loc[primary_rows["split"].eq("train"), "ID"].tolist()
expected_test_ids = primary_rows.loc[primary_rows["split"].eq("test"), "ID"].tolist()
validation_df = pd.concat([
    validate_materialized_root(FILTERED_TRAIN, "train", expected_train_ids),
    validate_materialized_root(FILTERED_TEST, "test", expected_test_ids),
], ignore_index=True)
validation_df.to_csv(DRIVER_TABLES / "filtered_root_validation.csv", index=False)

print("Audit-filtered paired cohort:")
print(primary_rows.groupby("split")["ID"].nunique().to_string())
print(f"Filtered train root: {FILTERED_TRAIN}")
print(f"Filtered test root:  {FILTERED_TEST}")
print(validation_df[[
    "split", "expected_materials", "Li_PDOS_files_found",
    "Total_DOS_files_found", "paired_materials_found"
]].to_string(index=False))
print("STAGING VALIDATION PASSED: real CSV files are present under /content.")
print("First staged training files:")
for _p in sorted(FILTERED_TRAIN.rglob("*.csv"))[:6]:
    print("  ", _p, _p.stat().st_size, "bytes")

# ========================== LOCATE OPTIONAL MANIFESTS =======================
def first_real_manifest(candidates, required_columns, placeholder_tokens=("replace_", "/path/to/")):
    for candidate in candidates:
        p = Path(candidate)
        if not p.exists():
            continue
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        if not required_columns.issubset(df.columns) or df.empty:
            continue
        joined = " ".join(df.astype(str).head(5).to_numpy().ravel()).lower()
        if any(token in joined for token in placeholder_tokens):
            continue
        return p
    return None

REFERENCE_MANIFEST = first_real_manifest([
    "/content/drive/MyDrive/MLIP-main/reference_spectra_manifest.csv",
    "/content/drive/MyDrive/OBELiX_reviewer_inputs/reference_spectra_manifest.csv",
    "/content/drive/MyDrive/OBELiX_convergence_outputs/reference_spectra_manifest.csv",
], {"case_id", "model_file", "reference_file"})

CONVERGENCE_MANIFEST = first_real_manifest([
    "/content/drive/MyDrive/OBELiX_convergence_outputs/convergence_manifest.csv",
    "/content/drive/MyDrive/MLIP-main/convergence_manifest.csv",
    "/content/convergence_manifest.csv",
], {"ID", "cif_path"})

# Resolve reference files robustly. The repository manifest stores the original
# Google Drive paths, but those files may instead be present inside the cloned
# repository's reference_spectra_preparation_outputs.zip. Extract that archive
# and rebase missing paths by filename so the comparison does not silently skip.
REFERENCE_ASSET_ROOT = Path("/content/obelix_reference_assets")
REFERENCE_ASSET_ROOT.mkdir(parents=True, exist_ok=True)


def extract_reference_asset_archives():
    archives = [
        Path("/content/drive/MyDrive/MLIP-main/reference_spectra_preparation_outputs.zip"),
        Path("/content/drive/MyDrive/MLIP-main/reference_spectra_preparation.zip"),
        Path("/content/drive/MyDrive/OBELiX_reviewer_inputs/reference_spectra_preparation_outputs.zip"),
    ]
    extracted = []
    for archive in archives:
        if not archive.exists():
            continue
        marker = REFERENCE_ASSET_ROOT / (archive.stem + ".extracted")
        if not marker.exists():
            with zipfile.ZipFile(archive, "r") as zf:
                zf.extractall(REFERENCE_ASSET_ROOT / archive.stem)
            marker.write_text(str(archive), encoding="utf-8")
        extracted.append(str(archive))
    return extracted


REFERENCE_ARCHIVES_EXTRACTED = extract_reference_asset_archives()


def locate_reference_asset(original):
    p = Path(str(original))
    if p.exists() and p.is_file():
        return p
    basename = p.name
    roots = [
        Path("/content/drive/MyDrive/OBELiX_reviewer_inputs/reference_spectra_preparation"),
        Path("/content/drive/MyDrive/MLIP-main/reference_spectra_preparation"),
        REFERENCE_ASSET_ROOT,
    ]
    hits = []
    for root in roots:
        if root.exists():
            hits.extend(root.rglob(basename))
    hits = sorted({h.resolve() for h in hits if h.is_file()}, key=lambda h: (len(h.parts), str(h)))
    return hits[0] if hits else None


REFERENCE_RESOLUTION_TABLE = pd.DataFrame()
if REFERENCE_MANIFEST is not None:
    _rm = pd.read_csv(REFERENCE_MANIFEST)
    _resolved_rows = []
    for _, _row in _rm.iterrows():
        _d = _row.to_dict()
        _model = locate_reference_asset(_d.get("model_file", ""))
        _reference = locate_reference_asset(_d.get("reference_file", ""))
        _d["model_file_original"] = str(_d.get("model_file", ""))
        _d["reference_file_original"] = str(_d.get("reference_file", ""))
        _d["model_file"] = str(_model) if _model is not None else ""
        _d["reference_file"] = str(_reference) if _reference is not None else ""
        _d["model_file_found"] = _model is not None
        _d["reference_file_found"] = _reference is not None
        _d["reference_pair_ready"] = bool(_model is not None and _reference is not None)
        _resolved_rows.append(_d)
    REFERENCE_RESOLUTION_TABLE = pd.DataFrame(_resolved_rows)
    REFERENCE_RESOLUTION_TABLE.to_csv(DRIVER_TABLES / "reference_manifest_file_resolution.csv", index=False)
    _ready = REFERENCE_RESOLUTION_TABLE[REFERENCE_RESOLUTION_TABLE["reference_pair_ready"]].copy()
    if len(_ready):
        _rebased = Path("/content/reference_spectra_manifest_rebased.csv")
        original_columns = [c for c in _rm.columns if c in _ready.columns]
        _ready[original_columns].to_csv(_rebased, index=False)
        REFERENCE_MANIFEST = _rebased
    else:
        REFERENCE_MANIFEST = None

# Empty nonexistent paths make the embedded engine skip optional analyses safely.
REFERENCE_MANIFEST_STR = str(REFERENCE_MANIFEST or "/content/no_reference_manifest.csv")
CONVERGENCE_MANIFEST_STR = str(CONVERGENCE_MANIFEST or "/content/no_convergence_manifest.csv")
print(f"Reference manifest:   {REFERENCE_MANIFEST_STR}")
if len(REFERENCE_RESOLUTION_TABLE):
    print("Reference pairs resolved:", int(REFERENCE_RESOLUTION_TABLE["reference_pair_ready"].sum()),
          "/", len(REFERENCE_RESOLUTION_TABLE))
print(f"Convergence manifest: {CONVERGENCE_MANIFEST_STR}")

# ==================== LOAD AND PATCH REPOSITORY ANALYSIS ENGINE =============
if not MASTER_SCRIPT.exists():
    import requests
    url = "https://raw.githubusercontent.com/Gajendra9843/MLIP/main/obelix_reviewer_response_onecell_code_2.py"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    MASTER_SCRIPT.write_text(response.text, encoding="utf-8")

source = MASTER_SCRIPT.read_text(encoding="utf-8")
marker = "# OBELiX REVIEWER-RESPONSE MASTER WORKFLOW"
position = source.find(marker)
if position < 0:
    raise RuntimeError(f"Could not locate master-workflow marker in {MASTER_SCRIPT}")
# Drop the separate reference-preparation program at the beginning. The final
# audit-filtered analysis engine begins at the master marker.
source = source[position:]

# Exact configuration replacements.
replacements = {
    'FINAL_OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/OBELiX_convergence_outputs"':
        f'FINAL_OUTPUT_DRIVE_DIR = r"{OUTPUT_ROOT}"',
    'FINAL_ZIP_NAME = "OBELiX_convergence_outputs.zip"':
        'FINAL_ZIP_NAME = "OBELiX_audit_filtered_publication_outputs.zip"',
    'CONVERGENCE_MANIFEST = "/content/convergence_manifest.csv"':
        f'CONVERGENCE_MANIFEST = r"{CONVERGENCE_MANIFEST_STR}"',
    'REFERENCE_MANIFEST = "/content/reference_spectra_manifest.csv"':
        f'REFERENCE_MANIFEST = r"{REFERENCE_MANIFEST_STR}"',
    'RUN_REVIEWER_SCALAR_BASELINES = False': 'RUN_REVIEWER_SCALAR_BASELINES = True',
    'RUN_REVIEWER_FREQUENCY_WARP = False': 'RUN_REVIEWER_FREQUENCY_WARP = True',
    'RUN_REVIEWER_STABILITY_SENSITIVITY = False': 'RUN_REVIEWER_STABILITY_SENSITIVITY = True',
    'RUN_REVIEWER_TOBIT_KERNEL = False': 'RUN_REVIEWER_TOBIT_KERNEL = True',
    'RUN_REVIEWER_FUSION_ABLATIONS = False': 'RUN_REVIEWER_FUSION_ABLATIONS = True',
    'RUN_REVIEWER_FAMILY_ANALYSES = False': 'RUN_REVIEWER_FAMILY_ANALYSES = True',
    'RUN_OPTIONAL_REFERENCE_COMPARISONS = False':
        f'RUN_OPTIONAL_REFERENCE_COMPARISONS = {bool(RUN_REFERENCE_VALIDATION and REFERENCE_MANIFEST is not None)}',
    'RUN_OPTIONAL_PHONON_CONVERGENCE = True   # runs only when manifest exists':
        f'RUN_OPTIONAL_PHONON_CONVERGENCE = {bool(RUN_PHONON_CONVERGENCE_IF_MANIFEST_EXISTS and CONVERGENCE_MANIFEST is not None)}',
    'REVIEWER_RANDOM_SEED = 20260626': f'REVIEWER_RANDOM_SEED = {RANDOM_SEED}',
    'REVIEWER_BOOTSTRAP = 3000': f'REVIEWER_BOOTSTRAP = {REVIEWER_BOOTSTRAP}',
    'RANDOM_SEED = 42': f'RANDOM_SEED = {RANDOM_SEED}',
    'N_BOOTSTRAP = 500': f'N_BOOTSTRAP = {ASSOCIATION_BOOTSTRAP}',
    'N_PERMUTATIONS = 1000': f'N_PERMUTATIONS = {ASSOCIATION_PERMUTATIONS}',
    'N_MI_PERMUTATIONS = 100': f'N_MI_PERMUTATIONS = {ASSOCIATION_MI_PERMUTATIONS}',
    'RANDOM_SEED = 20260624': f'RANDOM_SEED = {RANDOM_SEED}',
    'N_BOOTSTRAP = 3000': f'N_BOOTSTRAP = {KERNEL_BOOTSTRAP}',
    'N_HSIC_PERMUTATIONS = 5000': f'N_HSIC_PERMUTATIONS = {HSIC_PERMUTATIONS}',
    'INPUT_RESULTS_DIR = None  # optionally point directly to an extracted results directory':
        'INPUT_RESULTS_DIR = "/content/obelix_dos_pdos_association/results"',
}
for old, new in replacements.items():
    source = source.replace(old, new)

# Replace every embedded spectrum-root assignment, including repeated sections.
source = re.sub(
    r'TRAIN_ROOT\s*=\s*Path\([^\n]+\)',
    f'TRAIN_ROOT = Path(r"{FILTERED_TRAIN}")',
    source,
)
source = re.sub(
    r'TEST_ROOT\s*=\s*Path\([^\n]+\)',
    f'TEST_ROOT = Path(r"{FILTERED_TEST}")',
    source,
)
source = re.sub(
    r'TRAIN_PDOS_ROOT\s*=\s*Path\([^\n]+\)',
    f'TRAIN_PDOS_ROOT = Path(r"{FILTERED_TRAIN}")',
    source,
)
source = re.sub(
    r'TEST_PDOS_ROOT\s*=\s*Path\([^\n]+\)',
    f'TEST_PDOS_ROOT = Path(r"{FILTERED_TEST}")',
    source,
)

# Suppress every legacy figure generated by the embedded engine. The numerical
# analyses and CSV tables still run. Publication plots are regenerated below,
# each with an exact companion CSV.
style_patch = r'''
from matplotlib.figure import Figure as _OBELIX_Figure
from matplotlib.axes import Axes as _OBELIX_Axes
if not hasattr(_OBELIX_Figure, "_obelix_original_savefig"):
    _OBELIX_Figure._obelix_original_savefig = _OBELIX_Figure.savefig
if not hasattr(plt, "_obelix_original_savefig"):
    plt._obelix_original_savefig = plt.savefig
_OBELIX_Figure.savefig = lambda self, *args, **kwargs: None
plt.savefig = lambda *args, **kwargs: None
plt.title = lambda *args, **kwargs: None
_OBELIX_Axes.set_title = lambda self, *args, **kwargs: None
plt.rcParams.update({
    "font.size": 12, "axes.labelsize": 14, "xtick.labelsize": 11.5,
    "ytick.labelsize": 11.5, "legend.fontsize": 10.5,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
'''
source = source.replace("import matplotlib.pyplot as plt", "import matplotlib.pyplot as plt\n" + style_patch)

# Do not trigger browser downloads from intermediate or final engine stages.
source = source.replace("files.download(str(output_zip))", "print('Intermediate archive:', output_zip)")
source = source.replace("files.download(str(final_zip))", "print('Final local archive:', final_zip)")

patched_script = LOGS / "patched_audit_filtered_master_workflow.py"
patched_script.write_text(source, encoding="utf-8")
compile(source, str(patched_script), "exec")
print(f"Executing patched workflow: {patched_script}")

# The engine runs association, distribution-aware kernels, nested grouped CV,
# paired test bootstrap, HSIC, family transfer, censoring sensitivity, scalar
# baselines, frequency warps, stability filtering, censored KRR, fusion
# ablations, family analyses, and optional reference/convergence analyses.
exec(compile(source, str(patched_script), "exec"), globals(), globals())

# Restore normal figure saving for the curated publication exporter.
Figure.savefig = Figure._obelix_original_savefig
if hasattr(plt, "_obelix_original_savefig"):
    plt.savefig = plt._obelix_original_savefig
# Keep titles disabled by policy.
plt.title = lambda *args, **kwargs: None
Axes.set_title = lambda self, *args, **kwargs: None

# ======================= CURATED PUBLICATION EXPORTER =======================
# Every saved plot uses export_plot(), which writes the exact plot data CSV,
# vector PDF, and 600-dpi PNG under parallel directories.
CB = {
    "blue": "#0072B2", "orange": "#E69F00", "green": "#009E73",
    "red": "#D55E00", "purple": "#CC79A7", "sky": "#56B4E9",
    "black": "#222222", "gray": "#777777",
}

def panel_label(ax, label):
    ax.text(-0.14, 1.04, label, transform=ax.transAxes, fontsize=15,
            fontweight="bold", va="bottom", ha="left", clip_on=False)

def beautify(ax, grid_axis=None):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=4, width=1)
    if grid_axis:
        ax.grid(axis=grid_axis, alpha=0.20, linewidth=0.7)
    ax.margins(x=0.03, y=0.08)

def export_plot(fig, stem, data, section="main", annotations=None):
    if section == "main":
        folder = MAIN_FIGURES
    elif section == "reference":
        folder = REFERENCE_FIGURES
    else:
        folder = SUPP_FIGURES
    folder.mkdir(parents=True, exist_ok=True)
    data_path = PLOT_DATA / f"{stem}.csv"
    if isinstance(data, pd.DataFrame):
        data.to_csv(data_path, index=False)
    else:
        pd.DataFrame(data).to_csv(data_path, index=False)
    if annotations is not None:
        pd.DataFrame(annotations).to_csv(PLOT_DATA / f"{stem}_annotations.csv", index=False)
    for ax in fig.axes:
        ax.set_title("")
    fig.savefig(folder / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(folder / f"{stem}.png", dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

# Helper to find one engine output table.
def find_output(filename):
    roots = [
        Path("/content/obelix_dos_pdos_association/results"),
        Path("/content/obelix_kernel_publication/results"),
        Path("/content/obelix_reviewer_response/results"),
        Path("/content/obelix_reviewer_complete"),
    ]
    hits = []
    for root in roots:
        if root.exists():
            hits.extend(root.rglob(filename))
    if not hits:
        return None
    hits = sorted(set(hits), key=lambda p: (len(p.parts), str(p)))
    return hits[0]

def read_output(filename):
    p = find_output(filename)
    if p is None:
        return None
    try:
        return pd.read_csv(p)
    except Exception:
        return None

# --------------------------- MAIN FIGURE 1 ----------------------------------
# Fig. 1a: dataset waterfall.
waterfall = pd.DataFrame([
    {"stage": "Official OBELiX", "train": int((metadata.split == "train").sum()), "test": int((metadata.split == "test").sum())},
    {"stage": "Paired spectra", "train": int(((cohort_manifest.split == "train") & cohort_manifest.has_paired_spectrum).sum()), "test": int(((cohort_manifest.split == "test") & cohort_manifest.has_paired_spectrum).sum())},
    {"stage": "Primary cohort", "train": int(((cohort_manifest.split == "train") & cohort_manifest.included_primary).sum()), "test": int(((cohort_manifest.split == "test") & cohort_manifest.included_primary).sum())},
    {"stage": "Strict cohort", "train": int(((cohort_manifest.split == "train") & cohort_manifest.included_strict).sum()), "test": int(((cohort_manifest.split == "test") & cohort_manifest.included_strict).sum())},
    {"stage": "Exact cohort", "train": int(((cohort_manifest.split == "train") & cohort_manifest.included_exact).sum()), "test": int(((cohort_manifest.split == "test") & cohort_manifest.included_exact).sum())},
])
fig, ax = plt.subplots(figsize=(8.2, 5.3), constrained_layout=True)
x = np.arange(len(waterfall)); width = 0.38
ax.bar(x - width/2, waterfall.train, width, label="Train", color=CB["blue"])
ax.bar(x + width/2, waterfall.test, width, label="Test", color=CB["orange"])
ax.set_xticks(x); ax.set_xticklabels(waterfall.stage, rotation=22, ha="right")
ax.set_ylabel("Number of materials")
ax.legend(frameon=False, ncol=2, loc="upper right")
beautify(ax, "y"); panel_label(ax, "a")
export_plot(fig, "Fig1a_dataset_waterfall", waterfall, "main")

# Fig. 1b: family counts in the primary cohort.
family_counts = primary_rows.groupby(["Family", "split"]).size().unstack(fill_value=0).reset_index()
for c in ["train", "test"]:
    if c not in family_counts: family_counts[c] = 0
family_counts["total"] = family_counts.train + family_counts.test
family_counts = family_counts.sort_values("total")
fig, ax = plt.subplots(figsize=(8.5, max(5.2, 0.34 * len(family_counts))), constrained_layout=True)
y = np.arange(len(family_counts))
ax.barh(y, family_counts.train, label="Train", color=CB["blue"])
ax.barh(y, family_counts.test, left=family_counts.train, label="Test", color=CB["orange"])
ax.set_yticks(y); ax.set_yticklabels(family_counts.Family)
ax.set_xlabel("Number of materials")
ax.legend(frameon=False, ncol=2, loc="lower right")
beautify(ax, "x"); panel_label(ax, "b")
export_plot(fig, "Fig1b_family_counts", family_counts, "main")

# Fig. 1c: retained audit classes.
audit_counts = primary_rows.groupby(["composition_status", "split"]).size().unstack(fill_value=0).reset_index()
for c in ["train", "test"]:
    if c not in audit_counts: audit_counts[c] = 0
fig, ax = plt.subplots(figsize=(8.3, 4.8), constrained_layout=True)
y = np.arange(len(audit_counts))
ax.barh(y, audit_counts.train, label="Train", color=CB["blue"])
ax.barh(y, audit_counts.test, left=audit_counts.train, label="Test", color=CB["orange"])
ax.set_yticks(y); ax.set_yticklabels(audit_counts.composition_status.str.replace("_", " "))
ax.set_xlabel("Number of retained materials")
ax.legend(frameon=False, ncol=2, loc="lower right")
beautify(ax, "x"); panel_label(ax, "c")
export_plot(fig, "Fig1c_audit_status_counts", audit_counts, "main")

# Fig. 1d: conductivity coverage, exact values vs upper limits.
conductivity_plot = primary_rows[["split", "ID", "Family", "log10_ic_limit", "ic_is_upper_limit"]].dropna().copy()
fig, ax = plt.subplots(figsize=(7.6, 5.0), constrained_layout=True)
bins = np.linspace(conductivity_plot.log10_ic_limit.min()-0.2, conductivity_plot.log10_ic_limit.max()+0.2, 18)
for split, color in [("train", CB["blue"]), ("test", CB["orange"])]:
    vals = conductivity_plot.loc[(conductivity_plot.split == split) & ~conductivity_plot.ic_is_upper_limit, "log10_ic_limit"]
    ax.hist(vals, bins=bins, histtype="step", linewidth=2.0, label=f"{split.capitalize()} exact", color=color)
    lims = conductivity_plot.loc[(conductivity_plot.split == split) & conductivity_plot.ic_is_upper_limit, "log10_ic_limit"]
    ax.scatter(lims, np.zeros(len(lims))-0.35, marker="v", s=35, color=color, alpha=0.85, label=f"{split.capitalize()} upper limits")
ax.set_xlabel(r"$\log_{10}\sigma$ (S cm$^{-1}$)")
ax.set_ylabel("Number of materials")
ax.legend(frameon=False, ncol=2, loc="upper left")
beautify(ax, "y"); panel_label(ax, "d")
export_plot(fig, "Fig1d_conductivity_distribution", conductivity_plot, "main")

# --------------------------- MAIN FIGURE 2 ----------------------------------
def load_shape(split, representation, width="1THz"):
    return read_output(f"{split}_{representation}_shape_{width}.csv")

def shape_columns(df):
    if df is None: return []
    cols = []
    for c in df.columns:
        if re.search(r"_\d+\.\d+_\d+\.\d+THz$", str(c)):
            cols.append(c)
    return cols

def centers_from_cols(cols):
    out = []
    for c in cols:
        m = re.search(r"_(\d+\.\d+)_(\d+\.\d+)THz$", str(c))
        out.append((float(m.group(1)) + float(m.group(2))) / 2)
    return np.asarray(out)

def quartile_plot_data(rep):
    df = load_shape("train", rep, "1THz")
    if df is None: return None, None
    df["ID"] = df["ID"].map(normalize_id)
    cols = shape_columns(df); centers = centers_from_cols(cols)
    d = df.set_index("ID").join(train_metadata.set_index("ID")[["log10_ic_limit"]], how="inner")
    d = d.loc[d.index.intersection(set(primary_rows.loc[primary_rows.split == "train", "ID"]))]
    y = d.log10_ic_limit.to_numpy(float)
    q1, q3 = np.nanquantile(y, [0.25, 0.75])
    X = d[cols].apply(pd.to_numeric, errors="coerce").fillna(0).to_numpy(float)
    low = y <= q1; high = y >= q3
    rng = np.random.default_rng(RANDOM_SEED + (1 if rep == "li" else 2))
    boot = []
    for _ in range(ASSOCIATION_BOOTSTRAP):
        il = rng.choice(np.where(low)[0], low.sum(), replace=True)
        ih = rng.choice(np.where(high)[0], high.sum(), replace=True)
        boot.append(X[ih].mean(0) - X[il].mean(0))
    boot = np.asarray(boot)
    curve = pd.DataFrame({
        "frequency_THz": centers,
        "low_mean": X[low].mean(0),
        "high_mean": X[high].mean(0),
        "high_minus_low": X[high].mean(0) - X[low].mean(0),
        "difference_ci95_low": np.percentile(boot, 2.5, axis=0),
        "difference_ci95_high": np.percentile(boot, 97.5, axis=0),
        "low_quartile_threshold": q1,
        "high_quartile_threshold": q3,
        "n_low": int(low.sum()), "n_high": int(high.sum()),
    })
    curve["cumulative_high_minus_low"] = np.cumsum(curve.high_minus_low)
    return curve, d

li_curve, _ = quartile_plot_data("li")
total_curve, _ = quartile_plot_data("total")
if li_curve is not None:
    freq_mask = li_curve.frequency_THz <= 40
    p = li_curve[freq_mask]
    fig, ax = plt.subplots(figsize=(7.8, 5.1), constrained_layout=True)
    ax.plot(p.frequency_THz, p.low_mean, label="Lowest conductivity quartile", color=CB["blue"])
    ax.plot(p.frequency_THz, p.high_mean, label="Highest conductivity quartile", color=CB["red"])
    ax.set_xlabel("Frequency (THz)"); ax.set_ylabel("Normalized Li-PDOS weight")
    ax.legend(frameon=False, loc="upper right")
    beautify(ax); panel_label(ax, "a")
    export_plot(fig, "Fig2a_Li_PDOS_low_high_quartiles", p, "main")

    fig, ax = plt.subplots(figsize=(7.8, 5.1), constrained_layout=True)
    ax.axhline(0, color=CB["black"], linewidth=1)
    ax.fill_between(p.frequency_THz, p.difference_ci95_low, p.difference_ci95_high, color=CB["sky"], alpha=0.30, linewidth=0)
    ax.plot(p.frequency_THz, p.high_minus_low, color=CB["blue"])
    ax.set_xlabel("Frequency (THz)"); ax.set_ylabel("High − low Li-PDOS weight")
    beautify(ax); panel_label(ax, "b")
    export_plot(fig, "Fig2b_Li_PDOS_Q4_minus_Q1", p, "main")
if total_curve is not None:
    p = total_curve[total_curve.frequency_THz <= 40]
    fig, ax = plt.subplots(figsize=(7.8, 5.1), constrained_layout=True)
    ax.axhline(0, color=CB["black"], linewidth=1)
    ax.fill_between(p.frequency_THz, p.difference_ci95_low, p.difference_ci95_high, color=CB["orange"], alpha=0.25, linewidth=0)
    ax.plot(p.frequency_THz, p.high_minus_low, color=CB["red"])
    ax.set_xlabel("Frequency (THz)"); ax.set_ylabel("High − low total-DOS weight")
    beautify(ax); panel_label(ax, "c")
    export_plot(fig, "Fig2c_total_DOS_Q4_minus_Q1", p, "main")
if li_curve is not None and total_curve is not None:
    cdf = li_curve[["frequency_THz", "cumulative_high_minus_low"]].rename(columns={"cumulative_high_minus_low":"Li_PDOS"}).merge(
        total_curve[["frequency_THz", "cumulative_high_minus_low"]].rename(columns={"cumulative_high_minus_low":"total_DOS"}), on="frequency_THz")
    p = cdf[cdf.frequency_THz <= 40]
    fig, ax = plt.subplots(figsize=(7.8, 5.1), constrained_layout=True)
    ax.axhline(0, color=CB["black"], linewidth=1)
    ax.plot(p.frequency_THz, p.Li_PDOS, label="Li-PDOS", color=CB["blue"])
    ax.plot(p.frequency_THz, p.total_DOS, label="Total DOS", color=CB["red"])
    ax.set_xlabel("Frequency (THz)"); ax.set_ylabel("Cumulative high − low spectral weight")
    ax.legend(frameon=False, loc="best")
    beautify(ax); panel_label(ax, "d")
    export_plot(fig, "Fig2d_cumulative_spectral_difference", p, "main")

# --------------------------- MAIN FIGURE 3 ----------------------------------
def load_assoc(split, rep):
    return read_output(f"{split}_{rep}_shape_1THz_associations.csv")

for label, split, rep, panel, color in [
    ("Fig3a_Li_frequency_association_train", "train", "li", "a", CB["blue"]),
    ("Fig3b_Li_frequency_association_test", "test", "li", "b", CB["orange"]),
    ("Fig3c_total_frequency_association_train", "train", "total", "c", CB["green"]),
    ("Fig3d_total_frequency_association_test", "test", "total", "d", CB["red"]),
]:
    d = load_assoc(split, rep)
    if d is None or d.empty: continue
    d = d.sort_values("bin_center_THz")
    p = d[d.bin_center_THz <= 40]
    fig, ax = plt.subplots(figsize=(7.8, 5.0), constrained_layout=True)
    ax.axhline(0, color=CB["black"], linewidth=1)
    if {"spearman_ci95_low", "spearman_ci95_high"}.issubset(p.columns):
        ax.fill_between(p.bin_center_THz, p.spearman_ci95_low, p.spearman_ci95_high, color=color, alpha=0.18, linewidth=0)
    ax.plot(p.bin_center_THz, p.spearman_rho, color=color)
    if "spearman_p_maxT" in p.columns:
        sig = p.spearman_p_maxT < 0.05
        ax.scatter(p.loc[sig, "bin_center_THz"], p.loc[sig, "spearman_rho"], s=28, color=color, edgecolor=CB["black"], linewidth=0.4, zorder=4)
    ax.set_xlabel("Frequency-bin center (THz)")
    ax.set_ylabel(r"Spearman $\rho$ with $\log_{10}\sigma$")
    ax.set_ylim(-0.75, 0.75)
    beautify(ax); panel_label(ax, panel)
    export_plot(fig, label, p, "main")

# --------------------------- MAIN FIGURE 4 ----------------------------------
# Fig. 4a: compact descriptor train/test replication.
train_compact = read_output("train_compact_descriptor_associations.csv")
test_compact = read_output("test_compact_descriptor_associations.csv")
if train_compact is not None and test_compact is not None:
    descriptors = [
        "LiPDOS_fraction_below_2THz", "LiPDOS_fraction_below_5THz", "LiPDOS_q05_THz", "LiPDOS_centroid_THz",
        "totalDOS_fraction_below_2THz", "totalDOS_fraction_below_5THz", "totalDOS_q05_THz", "totalDOS_centroid_THz",
        "Li_fraction_below_2THz", "Li_fraction_below_5THz", "Li_q05_THz", "Li_centroid_THz",
        "total_fraction_below_2THz", "total_fraction_below_5THz", "total_q05_THz", "total_centroid_THz",
    ]
    tc = train_compact[train_compact.feature.isin(descriptors)].copy()
    ec = test_compact[test_compact.feature.isin(descriptors)].copy()
    repl = tc[["feature", "spearman_rho", "spearman_ci95_low", "spearman_ci95_high"]].rename(columns={
        "spearman_rho":"train_rho", "spearman_ci95_low":"train_low", "spearman_ci95_high":"train_high"}).merge(
        ec[["feature", "spearman_rho", "spearman_ci95_low", "spearman_ci95_high"]].rename(columns={
        "spearman_rho":"test_rho", "spearman_ci95_low":"test_low", "spearman_ci95_high":"test_high"}), on="feature", how="inner")
    repl["display"] = repl.feature.str.replace("_", " ").str.replace("THz", " THz")
    repl = repl.sort_values("test_rho")
    y = np.arange(len(repl))
    fig, ax = plt.subplots(figsize=(9.2, 6.0), constrained_layout=True)
    ax.axvline(0, color=CB["black"], linewidth=1)
    ax.errorbar(repl.train_rho, y-0.12, xerr=[repl.train_rho-repl.train_low, repl.train_high-repl.train_rho], fmt="o", capsize=3, color=CB["blue"], label="Train")
    ax.errorbar(repl.test_rho, y+0.12, xerr=[repl.test_rho-repl.test_low, repl.test_high-repl.test_rho], fmt="s", capsize=3, color=CB["orange"], label="Test")
    ax.set_yticks(y); ax.set_yticklabels(repl.display)
    ax.set_xlabel(r"Spearman $\rho$ with $\log_{10}\sigma$")
    ax.legend(frameon=False, ncol=2, loc="lower right")
    beautify(ax, "x"); panel_label(ax, "a")
    export_plot(fig, "Fig4a_compact_descriptor_replication", repl, "main")

# Fig. 4b: marginal/family-demeaned/static-residual associations where available.
adjusted = read_output("family_demeaned_scalar_associations.csv")
conditional = read_output("conditional_residual_associations.csv")
plot_adjusted = []
if adjusted is not None:
    for _, r in adjusted.iterrows():
        plot_adjusted.append({"split": r.get("split"), "descriptor": r.get("descriptor"), "adjustment":"Family-demeaned", "rho":r.get("spearman_rho"), "low":r.get("ci95_low"), "high":r.get("ci95_high")})
if conditional is not None and not conditional.empty:
    cc = conditional.copy()
    if "bin_center_THz" in cc.columns:
        cc = cc[pd.to_numeric(cc["bin_center_THz"], errors="coerce") <= 40]
    if "representation" in cc.columns:
        cc = cc[cc["representation"].astype(str).str.contains("li", case=False, regex=False)]
    cc["abs_rho"] = pd.to_numeric(cc.get("spearman_rho", np.nan), errors="coerce").abs()
    cc = cc.sort_values("abs_rho", ascending=False).groupby(cc.get("split", pd.Series("train", index=cc.index))).head(4)
    for _, r in cc.iterrows():
        plot_adjusted.append({"split": r.get("split", "train"), "descriptor": r.get("feature", r.get("descriptor", "")), "adjustment":"Static residual", "rho":r.get("spearman_rho", r.get("rho", np.nan)), "low":r.get("spearman_ci95_low", np.nan), "high":r.get("spearman_ci95_high", np.nan)})
adjusted_plot = pd.DataFrame(plot_adjusted)
if not adjusted_plot.empty:
    adjusted_plot = adjusted_plot[np.isfinite(pd.to_numeric(adjusted_plot.rho, errors="coerce"))].copy()
    adjusted_plot["label"] = adjusted_plot.descriptor.astype(str).str.replace("_", " ") + " | " + adjusted_plot.adjustment + " | " + adjusted_plot.split.astype(str)
    adjusted_plot = adjusted_plot.sort_values("rho")
    fig, ax = plt.subplots(figsize=(9.5, max(5.0, 0.34*len(adjusted_plot))), constrained_layout=True)
    y = np.arange(len(adjusted_plot)); ax.axvline(0, color=CB["black"], linewidth=1)
    ax.scatter(adjusted_plot.rho, y, color=CB["purple"], s=42)
    good = np.isfinite(pd.to_numeric(adjusted_plot.low, errors="coerce")) & np.isfinite(pd.to_numeric(adjusted_plot.high, errors="coerce"))
    if good.any():
        ax.errorbar(adjusted_plot.loc[good,"rho"], y[good], xerr=[adjusted_plot.loc[good,"rho"]-adjusted_plot.loc[good,"low"], adjusted_plot.loc[good,"high"]-adjusted_plot.loc[good,"rho"]], fmt="none", capsize=3, color=CB["purple"])
    ax.set_yticks(y); ax.set_yticklabels(adjusted_plot.label)
    ax.set_xlabel(r"Adjusted Spearman $\rho$")
    beautify(ax, "x"); panel_label(ax, "b")
    export_plot(fig, "Fig4b_adjusted_descriptor_associations", adjusted_plot, "main")

# Fig. 4c: HSIC summary.
hsic = read_output("HSIC_dependence_tests.csv")
if hsic is not None and not hsic.empty:
    hs = hsic.copy()
    hs["label"] = hs["component"].astype(str).str.replace("_", " ") + " | " + hs["permutation_scheme"].astype(str).str.replace("_", " ")
    # Prefer Wasserstein, static, within-family, and residual rows.
    keep = hs.component.astype(str).str.contains("wasserstein|static", case=False, regex=True)
    hs = hs[keep].copy().sort_values("normalized_HSIC" if "normalized_HSIC" in hs.columns else "HSIC")
    value_col = "normalized_HSIC" if "normalized_HSIC" in hs.columns else "HSIC"
    fig, ax = plt.subplots(figsize=(9.2, max(5.0, 0.34*len(hs))), constrained_layout=True)
    y = np.arange(len(hs)); ax.barh(y, hs[value_col], color=CB["green"])
    ax.set_yticks(y); ax.set_yticklabels(hs.label)
    ax.set_xlabel("Normalized HSIC")
    beautify(ax, "x"); panel_label(ax, "c")
    export_plot(fig, "Fig4c_HSIC_summary", hs, "main")

# Fig. 4d: Li-versus-total Wasserstein HSIC comparison.
hsic_diff = read_output("HSIC_Li_vs_total_Wasserstein.csv")
if hsic_diff is not None and not hsic_diff.empty:
    row = hsic_diff.iloc[0]
    cols = [c for c in hsic_diff.columns if "hsic" in c.lower()]
    # Construct stable plotting data from available columns.
    plot_rows = []
    for c in cols:
        if "difference" not in c.lower() and "delta" not in c.lower() and np.isscalar(row[c]):
            try: plot_rows.append({"quantity":c, "value":float(row[c])})
            except Exception: pass
    if not plot_rows:
        for c in cols:
            try: plot_rows.append({"quantity":c, "value":float(row[c])})
            except Exception: pass
    hplot = pd.DataFrame(plot_rows)
    if not hplot.empty:
        fig, ax = plt.subplots(figsize=(7.8, 4.8), constrained_layout=True)
        ax.barh(np.arange(len(hplot)), hplot.value, color=CB["purple"])
        ax.set_yticks(np.arange(len(hplot))); ax.set_yticklabels(hplot.quantity.str.replace("_", " "))
        ax.set_xlabel("HSIC statistic")
        beautify(ax, "x"); panel_label(ax, "d")
        export_plot(fig, "Fig4d_HSIC_Li_vs_total", hplot, "main", hsic_diff.to_dict("records"))

# --------------------------- MAIN FIGURE 5 ----------------------------------
perf = read_output("heldout_bootstrap_metric_intervals.csv")
preds = read_output("all_kernel_predictions.csv")
paired = read_output("paired_bootstrap_differences_vs_static.csv")
if perf is not None and not perf.empty:
    p = perf[perf.metric.eq("MAE")].copy().sort_values("point" if "point" in perf.columns else "estimate", ascending=False)
    point_col = "point" if "point" in p.columns else "estimate"
    label_map = {}
    # Retrieve labels from performance summary if available.
    summary = read_output("kernel_performance_summary.csv")
    if summary is not None and {"model","label"}.issubset(summary.columns):
        label_map = dict(zip(summary.model, summary.label))
    p["display"] = p.model.map(label_map).fillna(p.model).str.replace("_", " ")
    y = np.arange(len(p))
    fig, ax = plt.subplots(figsize=(9.3, max(5.4, 0.38*len(p))), constrained_layout=True)
    ax.errorbar(p[point_col], y, xerr=[p[point_col]-p.ci95_low, p.ci95_high-p[point_col]], fmt="o", capsize=3, color=CB["blue"])
    ax.set_yticks(y); ax.set_yticklabels(p.display)
    ax.set_xlabel(r"Official-test MAE in $\log_{10}\sigma$")
    beautify(ax, "x"); panel_label(ax, "a")
    export_plot(fig, "Fig5a_model_performance", p, "main")

if preds is not None and not preds.empty:
    # Normalize split labels used by different engine versions.
    split_col = "split"
    test_values = {"test", "official_test"}
    parity_models = [
        ("static_rbf", "Fig5b_static_parity", "b", CB["gray"]),
        ("li_wasserstein", "Fig5c_Li_Wasserstein_parity", "c", CB["blue"]),
        ("static_plus_li_wasserstein", "Fig5d_static_plus_Li_parity", "d", CB["red"]),
    ]
    for model, stem, panel, color in parity_models:
        d = preds[(preds.model == model) & preds[split_col].astype(str).isin(test_values)].copy()
        if d.empty: continue
        obs = d.observed.to_numpy(float); pre = d.predicted.to_numpy(float)
        lim = [min(obs.min(), pre.min())-0.4, max(obs.max(), pre.max())+0.4]
        mae = mean_absolute_error(obs, pre); rmse = np.sqrt(mean_squared_error(obs, pre)); r2 = r2_score(obs, pre); rho = stats.spearmanr(obs, pre).statistic
        annotations = [{"model":model,"MAE":mae,"RMSE":rmse,"R2":r2,"Spearman_rho":rho,"n":len(d)}]
        fig, ax = plt.subplots(figsize=(5.9, 5.6), constrained_layout=True)
        cens = d.get("is_upper_limit", pd.Series(False, index=d.index)).fillna(False).astype(bool)
        ax.scatter(obs[~cens], pre[~cens], s=42, color=color, alpha=0.82, edgecolor="white", linewidth=0.45, label="Exact")
        if cens.any():
            ax.scatter(obs[cens], pre[cens], s=48, color=color, marker="v", alpha=0.9, edgecolor=CB["black"], linewidth=0.45, label="Upper limit")
        ax.plot(lim, lim, linestyle="--", color=CB["black"], linewidth=1.2)
        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(r"Observed $\log_{10}\sigma$")
        ax.set_ylabel(r"Predicted $\log_{10}\sigma$")
        text = f"MAE = {mae:.2f}\nRMSE = {rmse:.2f}\n$R^2$ = {r2:.2f}\n$\rho$ = {rho:.2f}"
        ax.text(0.04, 0.96, text, transform=ax.transAxes, va="top", ha="left", fontsize=10.5, bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="0.75", alpha=0.92))
        if cens.any(): ax.legend(frameon=False, loc="lower right")
        beautify(ax); panel_label(ax, panel)
        export_plot(fig, stem, d, "main", annotations)

if paired is not None and not paired.empty:
    if {"model", "baseline", "difference_model_minus_baseline"}.issubset(paired.columns):
        d = paired[(paired["model"].astype(str) == "static_plus_li_wasserstein") & (paired["baseline"].astype(str) == "static_rbf")].copy()
        point_col = "difference_model_minus_baseline"
    elif {"model_A", "model_B", "difference_A_minus_B"}.issubset(paired.columns):
        d = paired[(paired["model_A"].astype(str) == "static_plus_li_wasserstein") & (paired["model_B"].astype(str) == "static_rbf")].copy()
        point_col = "difference_A_minus_B"
    else:
        d = pd.DataFrame(); point_col = ""
    if not d.empty:
        d["display"] = d.metric.replace({"Spearman_rho":"Spearman ρ"})
        y = np.arange(len(d)); point = d[point_col]
        fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
        ax.axvline(0, color=CB["black"], linewidth=1)
        ax.errorbar(point, y, xerr=[point-d.ci95_low, d.ci95_high-point], fmt="o", capsize=4, color=CB["red"])
        ax.set_yticks(y); ax.set_yticklabels(d.display)
        ax.set_xlabel("Static + Li minus static")
        beautify(ax, "x"); panel_label(ax, "e")
        export_plot(fig, "Fig5e_paired_model_differences", d, "main")

# ------------------- EXPLICIT REFERENCE VALIDATION PACKAGE -------------------
# This section is independent of the embedded engine's legacy figures. It
# guarantees CSV-backed, title-free reference comparisons whenever the manifest
# and at least one file pair are available.

def _read_reference_curve(path, fcol, ycol):
    frame = pd.read_csv(path)
    if fcol not in frame.columns or ycol not in frame.columns:
        raise KeyError(f"Columns {fcol!r}, {ycol!r} not found in {path}")
    x = pd.to_numeric(frame[fcol], errors="coerce").to_numpy(float)
    y = pd.to_numeric(frame[ycol], errors="coerce").to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = np.clip(y[ok], 0, None)
    order = np.argsort(x)
    return x[order], y[order]


def _reference_curve_comparison(xm, ym, xr, yr):
    lo = max(0.0, float(np.min(xm)), float(np.min(xr)))
    hi = min(float(np.max(xm)), float(np.max(xr)))
    if hi <= lo:
        raise ValueError("No overlapping positive-frequency interval")
    grid = np.linspace(lo, hi, 2001)
    model = np.clip(np.interp(grid, xm, ym, left=0, right=0), 0, None)
    reference = np.clip(np.interp(grid, xr, yr, left=0, right=0), 0, None)
    model_area = float(np.trapezoid(model, grid))
    reference_area = float(np.trapezoid(reference, grid))
    if model_area <= 0 or reference_area <= 0:
        raise ValueError("Zero integrated intensity on common grid")
    model /= model_area; reference /= reference_area
    dx = np.diff(grid)
    model_cdf = np.concatenate([[0.0], np.cumsum(0.5 * (model[:-1] + model[1:]) * dx)])
    reference_cdf = np.concatenate([[0.0], np.cumsum(0.5 * (reference[:-1] + reference[1:]) * dx)])
    model_cdf /= model_cdf[-1]; reference_cdf /= reference_cdf[-1]
    w1 = float(np.trapezoid(np.abs(model_cdf - reference_cdf), grid))
    overlap = float(np.trapezoid(np.minimum(model, reference), grid))
    model_centroid = float(np.trapezoid(grid * model, grid))
    reference_centroid = float(np.trapezoid(grid * reference, grid))
    mask5 = grid <= 5.0
    model_f5 = float(np.trapezoid(model[mask5], grid[mask5])) if mask5.sum() > 1 else 0.0
    reference_f5 = float(np.trapezoid(reference[mask5], grid[mask5])) if mask5.sum() > 1 else 0.0
    model_q05 = float(np.interp(0.05, model_cdf, grid))
    reference_q05 = float(np.interp(0.05, reference_cdf, grid))
    curve = pd.DataFrame({
        "frequency_THz": grid,
        "MatterSim_Phonopy": model,
        "reference": reference,
        "MatterSim_CDF": model_cdf,
        "reference_CDF": reference_cdf,
    })
    metrics = {
        "common_min_THz": lo, "common_max_THz": hi,
        "W1_THz": w1, "overlap": overlap,
        "model_centroid_THz": model_centroid,
        "reference_centroid_THz": reference_centroid,
        "centroid_error_model_minus_reference_THz": model_centroid-reference_centroid,
        "model_fraction_below_5THz": model_f5,
        "reference_fraction_below_5THz": reference_f5,
        "delta_fraction_below_5THz": model_f5-reference_f5,
        "model_q05_THz": model_q05,
        "reference_q05_THz": reference_q05,
        "delta_q05_THz": model_q05-reference_q05,
    }
    return curve, metrics


reference_metric_rows = []
reference_error_rows = []
reference_case_plot_rows = []
if REFERENCE_MANIFEST is not None and Path(REFERENCE_MANIFEST).exists():
    ref_manifest = pd.read_csv(REFERENCE_MANIFEST)
    for _, row in ref_manifest.iterrows():
        case_id = str(row["case_id"])
        safe_case = re.sub(r"[^A-Za-z0-9._-]+", "_", case_id)[:120]
        try:
            xm, ym = _read_reference_curve(row["model_file"], row["model_frequency_column"], row["model_intensity_column"])
            xr, yr = _read_reference_curve(row["reference_file"], row["reference_frequency_column"], row["reference_intensity_column"])
            curve, metrics = _reference_curve_comparison(xm, ym, xr, yr)
            curve.insert(0, "case_id", case_id)
            curve.insert(1, "spectrum_label", row.get("spectrum_label", ""))
            curve.to_csv(REFERENCE_CASE_DATA / f"{safe_case}.csv", index=False)
            reference_metric_rows.append({
                "case_id": case_id,
                "reference_type": row.get("reference_type", ""),
                "spectrum_label": row.get("spectrum_label", ""),
                "model_file": row.get("model_file", ""),
                "reference_file": row.get("reference_file", ""),
                **metrics,
            })
            fig, ax = plt.subplots(figsize=(7.4, 5.0), constrained_layout=True)
            ax.plot(curve.frequency_THz, curve.MatterSim_Phonopy, label="MatterSim–Phonopy", color=CB["blue"])
            ax.plot(curve.frequency_THz, curve.reference, label="DFT reference", color=CB["orange"])
            ax.set_xlabel("Frequency (THz)")
            ax.set_ylabel("Unit-area spectral density")
            ax.legend(frameon=False, loc="best")
            beautify(ax)
            export_plot(fig, f"RefCase_{safe_case}", curve, "reference")
            reference_case_plot_rows.append({"case_id":case_id, "stem":f"RefCase_{safe_case}", "status":"generated"})
        except Exception as exc:
            reference_error_rows.append({"case_id":case_id, "error":str(exc),
                                         "model_file":row.get("model_file", ""),
                                         "reference_file":row.get("reference_file", "")})

reference_metrics = pd.DataFrame(reference_metric_rows)
reference_errors = pd.DataFrame(reference_error_rows)
reference_metrics.to_csv(REFERENCE_TABLES / "reference_validation_all_cases.csv", index=False)
reference_errors.to_csv(REFERENCE_TABLES / "reference_validation_errors.csv", index=False)
pd.DataFrame(reference_case_plot_rows).to_csv(REFERENCE_TABLES / "reference_case_plot_manifest.csv", index=False)

if not reference_metrics.empty:
    reference_metrics["display"] = reference_metrics["case_id"].astype(str).str.replace("_", " ")
    # W1 forest.
    d = reference_metrics.sort_values("W1_THz", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(9.6, max(5.4, 0.34*len(d))), constrained_layout=True)
    y = np.arange(len(d))
    colors = [CB["blue"] if str(x).lower().startswith("li") else CB["orange"] for x in d["spectrum_label"]]
    ax.barh(y, d.W1_THz, color=colors)
    ax.set_yticks(y); ax.set_yticklabels(d.display)
    ax.set_xlabel(r"Wasserstein distance $W_1$ (THz)")
    beautify(ax, "x")
    export_plot(fig, "FigR1_reference_W1_forest", d, "reference")

    # Spectral overlap forest.
    d2 = reference_metrics.sort_values("overlap").copy()
    fig, ax = plt.subplots(figsize=(9.6, max(5.4, 0.34*len(d2))), constrained_layout=True)
    y = np.arange(len(d2))
    colors = [CB["blue"] if str(x).lower().startswith("li") else CB["orange"] for x in d2["spectrum_label"]]
    ax.barh(y, d2.overlap, color=colors)
    ax.set_yticks(y); ax.set_yticklabels(d2.display)
    ax.set_xlabel("Spectral overlap")
    ax.set_xlim(0, 1)
    beautify(ax, "x")
    export_plot(fig, "FigR2_reference_overlap_forest", d2, "reference")

    # Centroid parity.
    fig, ax = plt.subplots(figsize=(6.4, 6.0), constrained_layout=True)
    for label, marker, color in [("Li-PDOS", "o", CB["blue"]), ("Total DOS", "s", CB["orange"])]:
        s = reference_metrics[reference_metrics.spectrum_label.astype(str).eq(label)]
        if len(s):
            ax.scatter(s.reference_centroid_THz, s.model_centroid_THz, marker=marker,
                       s=55, alpha=0.8, label=label, color=color, edgecolor=CB["black"], linewidth=0.4)
    vals = np.r_[reference_metrics.reference_centroid_THz, reference_metrics.model_centroid_THz]
    lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
    pad = max(0.4, 0.06*(hi-lo))
    ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], linestyle="--", color=CB["black"], linewidth=1)
    ax.set_xlim(lo-pad, hi+pad); ax.set_ylim(lo-pad, hi+pad)
    ax.set_xlabel("DFT-reference centroid (THz)")
    ax.set_ylabel("MatterSim–Phonopy centroid (THz)")
    ax.legend(frameon=False, loc="best")
    beautify(ax)
    export_plot(fig, "FigR3_reference_centroid_parity", reference_metrics, "reference")

    # Grouped metric summary.
    summary = reference_metrics.groupby("spectrum_label", dropna=False).agg(
        n=("case_id", "size"),
        mean_W1_THz=("W1_THz", "mean"),
        median_W1_THz=("W1_THz", "median"),
        mean_overlap=("overlap", "mean"),
        mean_centroid_error_THz=("centroid_error_model_minus_reference_THz", "mean"),
        mean_delta_fraction_below_5THz=("delta_fraction_below_5THz", "mean"),
        mean_delta_q05_THz=("delta_q05_THz", "mean"),
    ).reset_index()
    summary.to_csv(REFERENCE_TABLES / "reference_validation_summary_by_spectrum.csv", index=False)
    long = summary.melt(id_vars=["spectrum_label", "n"],
                        value_vars=["mean_W1_THz", "mean_overlap", "mean_centroid_error_THz",
                                    "mean_delta_fraction_below_5THz", "mean_delta_q05_THz"],
                        var_name="metric", value_name="value")
    long.to_csv(REFERENCE_TABLES / "reference_validation_summary_long.csv", index=False)
else:
    pd.DataFrame([{
        "status":"skipped", "reason":"No resolvable reference model/reference file pairs",
        "manifest":REFERENCE_MANIFEST_STR,
    }]).to_csv(REFERENCE_TABLES / "reference_validation_status.csv", index=False)

# ----------------------- SUPPLEMENTARY FIGURES -------------------------------
def generic_metric_plot(df, stem, xcol, labelcol, xlabel, section="supplement", color=None, zero=False):
    if df is None or df.empty or xcol not in df.columns or labelcol not in df.columns:
        return
    d = df.copy()
    d[xcol] = pd.to_numeric(d[xcol], errors="coerce")
    d = d[np.isfinite(d[xcol])].sort_values(xcol)
    if d.empty: return
    fig, ax = plt.subplots(figsize=(9.2, max(4.8, 0.34*len(d))), constrained_layout=True)
    if zero: ax.axvline(0, color=CB["black"], linewidth=1)
    ax.barh(np.arange(len(d)), d[xcol], color=color or CB["blue"])
    ax.set_yticks(np.arange(len(d))); ax.set_yticklabels(d[labelcol].astype(str).str.replace("_", " "))
    ax.set_xlabel(xlabel); beautify(ax, "x")
    export_plot(fig, stem, d, section)

# S1: censoring sensitivity.
censor = read_output("censoring_policy_kernel_sensitivity.csv")
if censor is not None:
    policy_col = "censor_policy" if "censor_policy" in censor.columns else "policy"
    if policy_col in censor.columns and "model" in censor.columns:
        censor["label"] = censor[policy_col].astype(str) + " | " + censor["model"].astype(str)
        generic_metric_plot(censor, "FigS1_censoring_model_sensitivity", "R2", "label", r"Official-test $R^2$", color=CB["orange"], zero=True)

# S2: leave-one-family-out.
lofo = read_output("leave_one_family_out_performance.csv")
if lofo is not None and not lofo.empty:
    d = lofo[lofo.Family.ne("ALL_ELIGIBLE_FAMILIES")].copy()
    if "MAE" in d.columns:
        d["label"] = d.Family.astype(str) + " | " + d.model.astype(str)
        generic_metric_plot(d, "FigS2_leave_one_family_out_MAE", "MAE", "label", r"Leave-one-family-out MAE", color=CB["green"])

# S3: additive-kernel selected weights.
weights = read_output("outer_fold_selected_kernel_weights.csv")
if weights is not None and not weights.empty:
    if {"model", "component", "weight"}.issubset(weights.columns):
        long = weights[[c for c in ["model", "outer_fold", "component", "weight"] if c in weights.columns]].copy()
    else:
        numeric_weights = [c for c in weights.columns if "weight" in c.lower()]
        long = weights.melt(id_vars=[c for c in ["model","outer_fold"] if c in weights.columns], value_vars=numeric_weights, var_name="component", value_name="weight") if numeric_weights else pd.DataFrame()
    if not long.empty:
        long["label"] = long["model"].astype(str) + " | " + long["component"].astype(str)
        summary_w = long.groupby("label").weight.agg(["mean","std","count"]).reset_index()
        fig, ax = plt.subplots(figsize=(9.0, max(4.8,0.34*len(summary_w))), constrained_layout=True)
        y=np.arange(len(summary_w)); ax.errorbar(summary_w["mean"], y, xerr=summary_w["std"].fillna(0), fmt="o", capsize=3, color=CB["purple"])
        ax.set_yticks(y); ax.set_yticklabels(summary_w.label.str.replace("_"," "))
        ax.set_xlabel("Selected additive-kernel weight"); ax.set_xlim(-0.05,1.05); beautify(ax,"x")
        export_plot(fig,"FigS3_additive_kernel_weights",summary_w,"supplement", long.to_dict("records"))

# S4: scalar baselines.
scalar = read_output("scalar_baseline_performance.csv")
if scalar is not None:
    d=scalar[scalar.evaluation.eq("official_test")].copy() if "evaluation" in scalar.columns else scalar.copy()
    generic_metric_plot(d,"FigS4_scalar_baseline_R2","R2","model",r"Official-test $R^2$",color=CB["sky"],zero=True)

# S5: frequency-warp sensitivity.
warp = read_output("frequency_warp_sensitivity.csv")
if warp is not None:
    generic_metric_plot(warp,"FigS5_frequency_warp_R2","R2","warp",r"Official-test $R^2$",color=CB["blue"],zero=True)

# S6: imaginary-mode filtering.
stability = read_output("imaginary_mode_exclusion_sensitivity.csv")
if stability is not None:
    d=stability[stability.get("status","").astype(str).eq("ok")].copy() if "status" in stability.columns else stability.copy()
    generic_metric_plot(d,"FigS6_stability_filter_R2","R2","rule",r"Official-test $R^2$",color=CB["red"],zero=True)

# S7: formal censored regression.
tobit = read_output("formal_censored_kernel_regression_performance.csv")
if tobit is not None:
    metric = "official_test_interval_aware_MAE"
    generic_metric_plot(tobit,"FigS7_censored_kernel_interval_MAE",metric,"model",r"Interval-aware test MAE",color=CB["orange"])

# S8: fusion/stacking ablations.
fusion = read_output("fusion_and_stacking_ablation_performance.csv")
if fusion is not None:
    generic_metric_plot(fusion,"FigS8_fusion_ablation_R2","R2","model",r"Official-test $R^2$",color=CB["green"],zero=True)

# S9: family-specific incremental Li information.
inc = read_output("per_family_incremental_Li_information.csv")
if inc is not None and not inc.empty:
    d=inc[inc.split.eq("official_test")].copy() if "split" in inc.columns else inc.copy()
    generic_metric_plot(d,"FigS9_family_incremental_Li_MAE","delta_MAE_static_plus_Li_minus_static","Family",r"$\Delta$MAE: static + Li minus static",color=CB["purple"],zero=True)

# S10: residual diagnostics for principal models.
if preds is not None and not preds.empty:
    test_values={"test","official_test"}
    d=preds[preds.split.astype(str).isin(test_values) & preds.model.isin(["static_rbf","li_wasserstein","static_plus_li_wasserstein"])].copy()
    if not d.empty:
        d["residual"] = d.predicted-d.observed
        fig,ax=plt.subplots(figsize=(7.8,5.2),constrained_layout=True)
        for model,color in [("static_rbf",CB["gray"]),("li_wasserstein",CB["blue"]),("static_plus_li_wasserstein",CB["red"])]:
            s=d[d.model.eq(model)]; ax.scatter(s.predicted,s.residual,s=34,alpha=0.68,label=model.replace("_"," "),color=color)
        ax.axhline(0,color=CB["black"],linewidth=1); ax.set_xlabel(r"Predicted $\log_{10}\sigma$"); ax.set_ylabel("Prediction residual")
        ax.legend(frameon=False,loc="best"); beautify(ax)
        export_plot(fig,"FigS10_model_residuals",d,"supplement")

# S11: bin-width association sensitivity, based on saved association tables.
bin_rows=[]
for width in ["0.5THz","1THz","2THz"]:
    d=read_output(f"train_li_shape_{width}_associations.csv")
    if d is not None and not d.empty:
        j=d.spearman_rho.abs().idxmax(); r=d.loc[j]
        bin_rows.append({"bin_width":width,"strongest_frequency_THz":r.get("bin_center_THz"),"strongest_train_rho":r.get("spearman_rho"),"maxT_p":r.get("spearman_p_maxT",np.nan)})
bin_df=pd.DataFrame(bin_rows)
if not bin_df.empty:
    fig,ax=plt.subplots(figsize=(6.8,4.6),constrained_layout=True)
    ax.plot(bin_df.bin_width,bin_df.strongest_train_rho,marker="o",color=CB["blue"])
    ax.axhline(0,color=CB["black"],linewidth=1); ax.set_xlabel("Frequency-bin width"); ax.set_ylabel("Strongest train Spearman ρ")
    beautify(ax)
    export_plot(fig,"FigS11_bin_width_sensitivity",bin_df,"supplement")

# S12: reference-validation status in the Supplement package.
reference_status = pd.DataFrame([{
    "manifest": REFERENCE_MANIFEST_STR,
    "resolved_pairs": int(len(reference_metrics)),
    "failed_pairs": int(len(reference_errors)),
    "reference_figure_directory": str(REFERENCE_FIGURES),
    "reference_table_directory": str(REFERENCE_TABLES),
}])
reference_status.to_csv(SUPPLEMENT_TABLES / "FigS12_reference_validation_status.csv", index=False)

# S13: convergence metrics if available.
conv=read_output("phonon_convergence_vs_production.csv")
if conv is not None and not conv.empty:
    d=conv[conv.spectrum.astype(str).str.lower().eq("li")].copy() if "spectrum" in conv.columns else conv.copy()
    d["label"]=d.get("ID","").astype(str)+" | "+d.get("config","").astype(str)
    generic_metric_plot(d,"FigS13_convergence_W1","W1_0_to_100THz","label","Wasserstein distance from production (THz)",color=CB["orange"])

# S14: audit-cohort compact-descriptor sensitivity using exported scalar tables.
train_scalar=read_output("train_all_scalar_phonon_descriptors.csv")
test_scalar=read_output("test_all_scalar_phonon_descriptors.csv")
cohort_sens=[]
if train_scalar is not None and test_scalar is not None:
    for split, sdf, mdf in [("train",train_scalar,train_metadata),("test",test_scalar,test_metadata)]:
        sdf=sdf.copy(); sdf["ID"]=sdf.ID.map(normalize_id); merged=sdf.merge(mdf[["ID","log10_ic_limit"]],on="ID",how="inner")
        for cohort in ["primary","strict","exact"]:
            ids=set(cohort_manifest.loc[(cohort_manifest.split==split)&cohort_manifest[f"included_{cohort}"],"ID"])
            dd=merged[merged.ID.isin(ids)]
            for descriptor in ["Li_fraction_below_5THz","Li_q05_THz","total_fraction_below_5THz","total_q05_THz"]:
                if descriptor not in dd.columns or len(dd)<5: continue
                rho=stats.spearmanr(dd[descriptor],dd.log10_ic_limit,nan_policy="omit").statistic
                cohort_sens.append({"split":split,"cohort":cohort,"descriptor":descriptor,"n":len(dd),"spearman_rho":rho})
cohort_sens=pd.DataFrame(cohort_sens)
if not cohort_sens.empty:
    cohort_sens["label"]=cohort_sens.descriptor.str.replace("_"," ")+" | "+cohort_sens.split+" | "+cohort_sens.cohort
    generic_metric_plot(cohort_sens,"FigS14_audit_cohort_descriptor_sensitivity","spearman_rho","label",r"Spearman $\rho$",color=CB["red"],zero=True)

# S15: static-residual frequency associations.
conditional = read_output("conditional_residual_associations.csv")
if conditional is not None and not conditional.empty:
    d = conditional[(conditional["bin_width_THz"] == 1.0) &
                    conditional["representation"].isin(["conditional_li_shape", "conditional_total_shape"])].copy()
    d = d[d["bin_center_THz"] <= 40].sort_values(["representation", "bin_center_THz"])
    if len(d):
        fig, ax = plt.subplots(figsize=(7.8, 5.1), constrained_layout=True)
        ax.axhline(0, color=CB["black"], linewidth=1)
        for rep, label, color in [("conditional_li_shape", "Li-PDOS", CB["blue"]),
                                  ("conditional_total_shape", "Total DOS", CB["orange"])]:
            s = d[d.representation.eq(rep)]
            ax.plot(s.bin_center_THz, s.spearman_rho, label=label, color=color)
        ax.set_xlabel("Frequency-bin center (THz)")
        ax.set_ylabel(r"Spearman $\rho$ with static-model residual")
        ax.set_ylim(-0.75, 0.75)
        ax.legend(frameon=False, loc="best")
        beautify(ax)
        export_plot(fig, "FigS15_static_residual_frequency_associations", d, "supplement")

# S16: all HSIC dependence tests.
hsic_all = read_output("HSIC_dependence_tests.csv")
if hsic_all is not None and not hsic_all.empty:
    d = hsic_all.copy()
    d["label"] = d["component"].astype(str) + " | " + d["permutation_scheme"].astype(str)
    generic_metric_plot(d, "FigS16_HSIC_all_tests", "normalized_HSIC", "label",
                        "Normalized HSIC", color=CB["sky"])

# S17: all official-test model performances.
perf_all = read_output("kernel_performance_summary.csv")
if perf_all is not None and not perf_all.empty:
    d = perf_all[perf_all["evaluation"].astype(str).eq("official_test")].copy()
    generic_metric_plot(d, "FigS17_all_kernel_model_R2", "R2", "label",
                        r"Official-test $R^2$", color=CB["blue"], zero=True)

# S18: train-to-test effect replication for frequency bins.
replication = read_output("train_test_effect_replication.csv")
if replication is not None and not replication.empty:
    d = replication[replication["representation"].astype(str).isin(["li_shape", "total_shape"])].copy()
    d = d[np.isfinite(pd.to_numeric(d["spearman_rho_train"], errors="coerce")) &
          np.isfinite(pd.to_numeric(d["spearman_rho_test"], errors="coerce"))]
    if len(d):
        fig, ax = plt.subplots(figsize=(6.6, 6.0), constrained_layout=True)
        for rep, label, color, marker in [("li_shape", "Li-PDOS", CB["blue"], "o"),
                                          ("total_shape", "Total DOS", CB["orange"], "s")]:
            s = d[d.representation.eq(rep)]
            ax.scatter(s.spearman_rho_train, s.spearman_rho_test, s=28, alpha=0.65,
                       label=label, color=color, marker=marker)
        ax.axhline(0, color=CB["black"], linewidth=0.8)
        ax.axvline(0, color=CB["black"], linewidth=0.8)
        ax.plot([-1, 1], [-1, 1], linestyle="--", color=CB["gray"], linewidth=1)
        ax.set_xlim(-0.8, 0.8); ax.set_ylim(-0.8, 0.8)
        ax.set_xlabel(r"Training Spearman $\rho$")
        ax.set_ylabel(r"Official-test Spearman $\rho$")
        ax.legend(frameon=False, loc="best")
        beautify(ax)
        export_plot(fig, "FigS18_train_test_effect_replication", d, "supplement")

# S19: spectrum-QC frequency coverage.
qc_all = read_output("spectrum_QC_all.csv")
if qc_all is not None and not qc_all.empty:
    d = qc_all[qc_all["status"].astype(str).eq("ok")].copy()
    if len(d):
        fig, ax = plt.subplots(figsize=(7.2, 5.0), constrained_layout=True)
        for mode, label, color, marker in [("pdos", "Li-PDOS", CB["blue"], "o"),
                                           ("dos", "Total DOS", CB["orange"], "s")]:
            s = d[d["mode"].astype(str).eq(mode)]
            ax.scatter(s.min_frequency_raw_THz, s.max_frequency_raw_THz,
                       s=25, alpha=0.55, label=label, color=color, marker=marker)
        ax.set_xlabel("Minimum raw frequency (THz)")
        ax.set_ylabel("Maximum raw frequency (THz)")
        ax.legend(frameon=False, loc="best")
        beautify(ax)
        export_plot(fig, "FigS19_spectrum_frequency_coverage", d, "supplement")

# S20: audit-class counts in all paired spectra, including exclusions.
audit_all = cohort_manifest[cohort_manifest["has_paired_spectrum"]].copy()
audit_all_counts = audit_all.groupby(["composition_status", "split"]).size().unstack(fill_value=0).reset_index()
for c in ["train", "test"]:
    if c not in audit_all_counts: audit_all_counts[c] = 0
fig, ax = plt.subplots(figsize=(9.0, max(4.8, 0.42*len(audit_all_counts))), constrained_layout=True)
y = np.arange(len(audit_all_counts))
ax.barh(y, audit_all_counts.train, label="Train", color=CB["blue"])
ax.barh(y, audit_all_counts.test, left=audit_all_counts.train, label="Test", color=CB["orange"])
ax.set_yticks(y); ax.set_yticklabels(audit_all_counts.composition_status.astype(str).str.replace("_", " "))
ax.set_xlabel("Number of paired spectra")
ax.legend(frameon=False, loc="lower right")
beautify(ax, "x")
export_plot(fig, "FigS20_all_paired_audit_statuses", audit_all_counts, "supplement")

# Explicit supplement-analysis manifest. Nothing is silently omitted.
EXPECTED_SUPPLEMENT_STEMS = [
    "FigS1_censoring_model_sensitivity", "FigS2_leave_one_family_out_MAE",
    "FigS3_additive_kernel_weights", "FigS4_scalar_baseline_R2",
    "FigS5_frequency_warp_R2", "FigS6_stability_filter_R2",
    "FigS7_censored_kernel_interval_MAE", "FigS8_fusion_ablation_R2",
    "FigS9_family_incremental_Li_MAE", "FigS10_model_residuals",
    "FigS11_bin_width_sensitivity", "FigS13_convergence_W1",
    "FigS14_audit_cohort_descriptor_sensitivity",
    "FigS15_static_residual_frequency_associations", "FigS16_HSIC_all_tests",
    "FigS17_all_kernel_model_R2", "FigS18_train_test_effect_replication",
    "FigS19_spectrum_frequency_coverage", "FigS20_all_paired_audit_statuses",
]
_supp_status = []
for stem in EXPECTED_SUPPLEMENT_STEMS:
    pdf = SUPP_FIGURES / f"{stem}.pdf"
    png = SUPP_FIGURES / f"{stem}.png"
    csv = PLOT_DATA / f"{stem}.csv"
    generated = pdf.exists() and png.exists() and csv.exists()
    reason = "generated" if generated else "source table unavailable, optional analysis skipped, or no valid rows"
    if stem == "FigS13_convergence_W1" and CONVERGENCE_MANIFEST is None:
        reason = "convergence_manifest.csv was not found; convergence calculations cannot be run"
    _supp_status.append({"stem":stem, "generated":generated, "reason":reason,
                         "pdf":str(pdf), "png":str(png), "csv":str(csv)})
pd.DataFrame(_supp_status).to_csv(OUTPUT_ROOT / "supplement_analysis_manifest.csv", index=False)

# Convergence status is always explicit.
pd.DataFrame([{
    "requested": bool(RUN_PHONON_CONVERGENCE_IF_MANIFEST_EXISTS),
    "manifest_found": CONVERGENCE_MANIFEST is not None,
    "manifest": CONVERGENCE_MANIFEST_STR,
    "status": "run_or_results_imported" if CONVERGENCE_MANIFEST is not None else "skipped_missing_manifest",
}]).to_csv(SUPPLEMENT_TABLES / "convergence_analysis_status.csv", index=False)

# ========================= OUTPUT COMPLETENESS AUDIT =========================
# Ensure each publication PDF/PNG has a plot-data CSV.
plot_rows=[]
for section, folder in [("main",MAIN_FIGURES),("supplement",SUPP_FIGURES),("reference",REFERENCE_FIGURES)]:
    stems=sorted({p.stem for p in folder.glob("*.pdf")} | {p.stem for p in folder.glob("*.png")})
    for stem in stems:
        plot_rows.append({
            "section":section,"stem":stem,
            "pdf_exists":(folder/f"{stem}.pdf").exists(),
            "png_exists":(folder/f"{stem}.png").exists(),
            "csv_exists":(PLOT_DATA/f"{stem}.csv").exists(),
            "annotations_csv_exists":(PLOT_DATA/f"{stem}_annotations.csv").exists(),
        })
plot_manifest=pd.DataFrame(plot_rows)
plot_manifest.to_csv(OUTPUT_ROOT/"publication_plot_manifest.csv",index=False)
if len(plot_manifest) and not plot_manifest[["pdf_exists","png_exists","csv_exists"]].all(axis=None):
    raise RuntimeError("At least one publication plot is missing a PDF, PNG, or companion CSV.")

# Copy the complete numerical results produced by the engine into the requested
# output directory, in addition to its combined ZIP.
for source_dir,name in [
    (Path("/content/obelix_dos_pdos_association/results"),"association_results"),
    (Path("/content/obelix_kernel_publication/results"),"kernel_results"),
    (Path("/content/obelix_reviewer_response/results"),"reviewer_response_results"),
]:
    if source_dir.exists():
        dest=OUTPUT_ROOT/name
        if dest.exists(): shutil.rmtree(dest)
        shutil.copytree(source_dir,dest)

# Reproducibility record.
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

run_record={
    "random_seed":RANDOM_SEED,
    "train_root":str(TRAIN_PDOS_ROOT),"test_root":str(TEST_PDOS_ROOT),
    "audit_csv":str(AUDIT_CSV),"audit_sha256":sha256(AUDIT_CSV),
    "master_script":str(MASTER_SCRIPT),"master_script_sha256":sha256(MASTER_SCRIPT),
    "primary_statuses":sorted(PRIMARY_COMPOSITION_STATUSES),
    "strict_statuses":sorted(STRICT_COMPOSITION_STATUSES),
    "exact_statuses":sorted(EXACT_COMPOSITION_STATUSES),
    "include_short_distance_warnings":INCLUDE_SHORT_DISTANCE_WARNINGS,
    "n_primary_train":int(((cohort_manifest.split=="train")&cohort_manifest.included_primary).sum()),
    "n_primary_test":int(((cohort_manifest.split=="test")&cohort_manifest.included_primary).sum()),
    "reference_manifest":REFERENCE_MANIFEST_STR,
    "reference_pairs_resolved":int(len(reference_metrics)),
    "reference_pairs_failed":int(len(reference_errors)),
    "convergence_manifest":CONVERGENCE_MANIFEST_STR,
    "figure_policy":"No titles; large fonts; PDF+600-dpi PNG+CSV for every publication panel",
}
(LOGS/"audit_filtered_run_record.json").write_text(json.dumps(run_record,indent=2),encoding="utf-8")

readme=f'''OBELiX audit-filtered publication analysis

Primary cohort statuses: {sorted(PRIMARY_COMPOSITION_STATUSES)}
Short-distance warnings included: {INCLUDE_SHORT_DISTANCE_WARNINGS}
Train materials: {run_record['n_primary_train']}
Test materials: {run_record['n_primary_test']}

publication_figures/main/       Main-text PDF and PNG panels
publication_figures/supplement/ Supplementary PDF and PNG panels
publication_figures/reference_validation/ Accepted-case reference overlays and summaries
reference_validation_tables/   Reference metrics, errors, and summary tables
supplement_tables/              Supplement-only status and auxiliary tables
plot_data/                      Exact CSV data used for every panel
publication_plot_manifest.csv   Completeness check for PDF/PNG/CSV triplets
supplement_analysis_manifest.csv Explicit generated/skipped status for each Supplement analysis
association_results/            Full association analysis tables
kernel_results/                 Nested-CV, prediction, bootstrap and HSIC tables
reviewer_response_results/      Reviewer-specific robustness analyses
'''
(OUTPUT_ROOT/"README_outputs.txt").write_text(readme,encoding="utf-8")

# Final archive containing everything under OUTPUT_ROOT.
archive_base=OUTPUT_ROOT.parent/"OBELiX_audit_filtered_publication_analysis"
archive_path=Path(shutil.make_archive(str(archive_base),"zip",root_dir=OUTPUT_ROOT))
print("\nAUDIT-FILTERED PUBLICATION ANALYSIS COMPLETE")
print(f"Results directory: {OUTPUT_ROOT}")
print(f"Archive: {archive_path}")
print(plot_manifest.to_string(index=False))


OBELiX analysis notebook version: 2026-07-19-v5-complete-supplement-reference
Mounted at /content/drive
Audit-filtered paired cohort:
split
test      48
train    212
Filtered train root: /content/obelix_audit_filtered_spectra/Train_cif
Filtered test root:  /content/obelix_audit_filtered_spectra/Test_cif
split  expected_materials  Li_PDOS_files_found  Total_DOS_files_found  paired_materials_found
train                 212                  212                    212                     212
 test                  48                   48                     48                      48
STAGING VALIDATION PASSED: real CSV files are present under /content.
First staged training files:
   /content/obelix_audit_filtered_spectra/Train_cif/00x/Lithium_PDOS_00x.csv 6763 bytes
   /content/obelix_audit_filtered_spectra/Train_cif/00x/Total_DOS_00x.csv 6476 bytes
   /content/obelix_audit_filtered_spectra/Train_cif/019/Lithium_PDOS_019.csv 6662 bytes
   /content/obelix_audit_filtered_spectra/Train_cif/0

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [05:30:58] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Models completed: DOS cohort / static only
Models completed: DOS cohort / total_shape
Models completed: DOS cohort / static + total_shape
Models completed: Li-PDOS cohort / static only
Models completed: Li-PDOS cohort / li_shape
Models completed: Li-PDOS cohort / static + li_shape
Models completed: paired cohort / static only
Models completed: paired cohort / total_shape+li_shape
Models completed: paired cohort / static + total_shape+li_shape

OBELiX DOS / Li-PDOS ASSOCIATION WORKFLOW
Device: Tesla T4
Frequency inclusion: 0 < frequency <= 100.0 THz
Negative-frequency structures were NOT removed; only non-positive frequency points were excluded from features.
Primary bin width: 1.0 THz; sensitivity widths: (0.5, 1.0, 2.0)
Primary target: log10_ic_limit

Coverage:
split  OBELiX_rows  valid_target_rows  parsed_DOS  parsed_Li_PDOS  paired_DOS_and_PDOS  DOS_with_target  PDOS_with_target  paired_with_target
train          478                478         212             212                  21

mattersim-v1.0.0-1M.pth: 100%|██████████| 17.1M/17.1M [00:00<00:00, 142MB/s]

2026-07-19 06:02:23.953 | INFO     | mattersim.utils.download_utils:download_file:44 - File downloaded to /root/.local/mattersim/pretrained_models/mattersim-v1.0.0-1M.pth
2026-07-19 06:02:23.955 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model


...

ValueError: 
$ho$ = 0.48
^
ParseException: Expected end of text, found '$'  (at char 0), (line:1, col:1)

Error in callback <function _draw_all_if_interactive at 0x7be2c0ed3240> (for post_execute):


ValueError: 
$ho$ = 0.48
^
ParseException: Expected end of text, found '$'  (at char 0), (line:1, col:1)

ValueError: 
$ho$ = 0.48
^
ParseException: Expected end of text, found '$'  (at char 0), (line:1, col:1)

<Figure size 590x560 with 1 Axes>